In [1]:
import galois

In [ ]:
ff = galois.GF( 2**8 )
print(ff.properties)

In [4]:
a = ff(3)
b = ff(7)

In [5]:
a*b
a+b
a-b
a/b

GF(211, order=2^8)

## Self-made

In [110]:
class Polynomial:
    def __init__( self, coefficient_list, variable="X" ):
        """[0,2,1,3] -> 2x^2+x+3"""
        assert len(coefficient_list)>0, "no coefficients"
        # thankfully multiplication and exponentiation with zero are defined as required
        zero = 0*coefficient_list[0]
        one = coefficient_list[0]**0
        while len(coefficient_list)>1:
            if coefficient_list[0] == zero:
                coefficient_list = coefficient_list[1:]
            else:
                break
        self.deg = len( coefficient_list ) - 1
        self.coefficients = coefficient_list
        self.zero = zero
        self.one = one
        self.variable = variable

    def __repr__( self ):
        zero = self.zero
        x = self.variable
        if self.coefficients == [ self.zero ]:
            return f"[({self.zero})]"
        output = ""
        if self.coefficients[-1] != zero:
            output += f"{self.coefficients[-1]}"
        for i in range( self.deg-1, -1, -1 ):
            if self.coefficients[i] != zero:
                if output == "":
                    output = f"({self.coefficients[i]})*" + self.variable + f"^{self.deg-i}"
                else:
                    output = f"({self.coefficients[i]})*" + self.variable + f"^{self.deg-i} + " + output
        return "[ " + output + " ]"

    def __eq__( self, other ):
        return self.coefficients == other.coefficients

    def leading_coefficient( self ):
        return self.coefficients[0]

    def normalize( self ):
        if self == Polynomial( [self.zero] ):
            return self
        else:
            lc = self.leading_coefficient()
            return Polynomial( [ lc**(-1) ] ) * self

    def __add__( self, other ):
        zero = self.zero
        deg1, deg2 = self.deg, other.deg
        maxdeg = max( deg1, deg2 )
        coefficients1 = (maxdeg-deg1)*[zero] + self.coefficients
        coefficients2 = (maxdeg-deg2)*[zero] + other.coefficients
        coefficients  = [ x+y for (x,y) in zip( coefficients1, coefficients2) ]
        return Polynomial( coefficients, variable=self.variable )

    def __sub__( self, other ):
        zero = self.zero
        deg1, deg2 = self.deg, other.deg
        maxdeg = max( deg1, deg2 )
        coefficients1 = (maxdeg-deg1)*[zero] + self.coefficients
        coefficients2 = (maxdeg-deg2)*[zero] + other.coefficients
        coefficients  = [ x-y for (x,y) in zip( coefficients1, coefficients2) ]
        return Polynomial( coefficients, variable=self.variable )

    def __mul__( self, other ):
        prod = Polynomial([self.zero], variable=self.variable)
        coeffs = self.coefficients
        for d in reversed(other.coefficients):
            mult_coeffs = [ d*c for c in coeffs ]
            prod = prod + Polynomial( mult_coeffs, variable=self.variable )
            coeffs = coeffs + [self.zero]
        return prod

    def __mod__( self, other ):
        _, r = Polynomial.quotient_remainder( self, other )
        return r

    def __floordiv__( self, other ):
        q, _ = Polynomial.quotient_remainder( self, other )
        return q

    def inverse_mod( self, other ):
        g,inverse,_ = Polynomial.extended_gcd( self, other )
        if g.deg > 1:
            raise ZeroDivisionError
        return inverse

    @classmethod
    def quotient_remainder( cls, a, b ):
#        return a._quotient_remainder( b )
        zero = a.zero
        blc = b.leading_coefficient()
        assert blc != zero, "division by zero"
        if b.deg == 0:
            return ( Polynomial( [blc**(-1)] ) * a, Polynomial( [zero] ) )
        d = b * Polynomial( [blc**(-1)] ) # use monic divisor
        # d = other.normalize()
        q = Polynomial( [zero] )
        r = a
        while r.deg >= d.deg:
            rlc = r.leading_coefficient()
            degdiff = r.deg - d.deg
            r = r - Polynomial([rlc]+degdiff*[zero]) * d
            q = q + Polynomial([rlc]+degdiff*[zero])
#        print( f"{q} Rest {r}")
        return Polynomial([blc**(-1)])*q, r

    @classmethod
    def extended_gcd( cls, a, b ):
#        print (f"\n a = {a}, b = {b}")
        zero = a.zero
        one = a.one
        assert a.leading_coefficient() != a.zero or b.leading_coefficient() != zero, "gcd undefined"
        if a.deg < b.deg:
            (g,x,y) = Polynomial.extended_gcd( b, a )
            return (g,y,x)
        x,y = Polynomial([zero]), Polynomial([one]) # 0,1
        u,v = Polynomial([one]), Polynomial([zero]) # 1,0
        while a != Polynomial([zero]): # 0:
#            q, r = b._quotient_remainder( a ) # q, r = b // a, b % a
            q, r = Polynomial.quotient_remainder( b, a ) # q, r = b // a, b % a
            m, n = x - (u*q), y - (v*q)
            b,a, x,y, u,v = a,r, u,v, m,n
#        print( f"{b} || {a} || {x} || {y} || {q}" )
        inv_lc = Polynomial( [(b.leading_coefficient())**(-1)] )
        return inv_lc * b, inv_lc * x, inv_lc * y


In [ ]:
a = Polynomial([1,0,0,0])
c = Polynomial([7,1,2])
q,r = Polynomial.quotient_remainder(a,c)
print( q, r )
print( a//c, a%c )
print( q*c+r == a )

[ (0.14285714285714285)*X^1 + -0.02040816326530612 ] [ (-0.26530612244897955)*X^1 + 0.04081632653061224 ]
[ (0.14285714285714285)*X^1 + -0.02040816326530612 ] [ (-0.26530612244897955)*X^1 + 0.04081632653061224 ]
True


In [96]:
p = Polynomial( [-1,-2,2,1,3],variable="x"  )
q = Polynomial( [1,1] )
r = p+q
print(r.coefficients)
print(p+q)
print(p-q)
r = p*q
print(r.coefficients)
s = Polynomial([0])
s.leading_coefficient()
#Polynomial.extended_gcd(p,q)

[-1, -2, 2, 2, 4]
[ (-1)*x^4 + (-2)*x^3 + (2)*x^2 + (2)*x^1 + 4 ]
[ (-1)*x^4 + (-2)*x^3 + (2)*x^2 + 2 ]
[-1, -3, 0, 3, 4, 3]


0

In [97]:
a = Polynomial( [2,1,1,1] )
b = Polynomial( [1,1,0] )
print (f"\n {b}, {a}")
print( Polynomial.extended_gcd( b, a ) )
print (f"\n {a}, {b}")
print( Polynomial.extended_gcd( a, b ) )



 [ (1)*X^2 + (1)*X^1 ], [ (2)*X^3 + (1)*X^2 + (1)*X^1 + 1 ]
([ 1.0 ], [ (-4.0)*X^2 + -3.0 ], [ (2.0)*X^1 + 1.0 ])

 [ (2)*X^3 + (1)*X^2 + (1)*X^1 + 1 ], [ (1)*X^2 + (1)*X^1 ]
([ 1.0 ], [ (2.0)*X^1 + 1.0 ], [ (-4.0)*X^2 + -3.0 ])


In [98]:
a = Polynomial( [2,1,1,1] )
b = Polynomial( [2,1,1,2,0] )
print( a.inverse_mod(b) )
print( a*a.inverse_mod(b) % b )

[ (2.0)*X^3 + (1.0)*X^2 + (1.0)*X^1 + 1.0 ]
[ 1.0 ]


In [99]:
Polynomial.extended_gcd(a,b)

([ 1.0 ],
 [ (2.0)*X^3 + (1.0)*X^2 + (1.0)*X^1 + 1.0 ],
 [ (-2.0)*X^2 + (-1.0)*X^1 + -1.0 ])

In [111]:
ff = galois.GF(2**8)

In [112]:
f = Polynomial( [ff(1),ff(0),ff(3),ff(2)])
f

[ (1)*X^3 + (α + 1)*X^1 + α ]

In [125]:
p = Polynomial( [ff(1),ff(0),ff(0),ff(0),ff(0),ff(0)])
p

[ (1)*X^5 ]

In [114]:
p % f

[ (α)*X^2 + (α^2 + 1)*X^1 + α^2 + α ]

In [115]:
pi=p.inverse_mod(f)

In [116]:
pi*p

[ (α^7 + α^6 + α^5 + α)*X^7 + (α^7 + α^6 + α^5 + α^4 + α^3 + α + 1)*X^6 + (α^4 + α^3)*X^5 ]

In [117]:
(pi*p)%f

[ 1 ]